In [0]:
# run_all.py

from scr.endpoints import list_keys_all, list_keys_changes, next_offset
from scr.connection import APIClient
from scr.loaders import fetch_all_keys_to_bronze, fetch_items_to_silver_json
from scr.curate_items import kuratoi_ja_talleta_deltaan_like_batch
import time
import pandas as pd  # <-- lisäys

# ----------------------------
# 0) Valitse ajotapa
# ----------------------------
# "full"    = koko katalogi (overwrite)
# "changes" = vain muutokset (append)
RUN_MODE = "full"

# ----------------------------
# 1) Asetukset
# ----------------------------
BASE = "https://productapi-synkka.gs1.fi"
EMAIL = dbutils.secrets.get("gs1-kv", "email")
PASSWORD = dbutils.secrets.get("gs1-kv", "password")
GLN = dbutils.secrets.get("gs1-kv", "gln")

account    = "gs1datalake"   # storage accountin nimi
container  = "datalake"      # containerin nimi
access_key = "5kEehpTCdCfzcmwOzmq+w4iaiFeMGnfV2OCaxQlGut2kb65ItOD4QDWapkmcT/NI4t8sLaOFAbjL+AStaIorWg=="

# Kerrotaan Sparkin asetuksissa, miten mennään storageen
spark.conf.set(f"fs.azure.account.key.{account}.dfs.core.windows.net", access_key)

# (suositus, auttaa skeemamuutoksiin append-ajossa)
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

# Delta-polut
DELTA_KEYS     = f"abfss://{container}@{account}.dfs.core.windows.net/gs1/bronze/public_item_sync"
DELTA_PRODUCTS = f"abfss://{container}@{account}.dfs.core.windows.net/gs1/silver/catalogue_items"
CURATED_ITEMS  = f"abfss://{container}@{account}.dfs.core.windows.net/gs1/curated/items_selected_fields"

# Parametrit
KEY_BATCH_SIZE = 90000
SINCE_ISO      = "2025-09-21T07:03:26.1655687Z"  # changes-leikkaus
ITEM_BATCH     = 1000                            # max 1000 / Many
RATE_LIMIT     = 15                              # Many-API: 15 pyyntöä / min

# GPC-segmenttisuodatin Silveriin vietäville riveille
ONLY_GPC_SEGMENT_CODE = "50000000"  # vaihda tarvittaessa tai aseta None, jos et halua suodatusta

# ----------------------------
# 2) Autentikointi
# ----------------------------
client = APIClient(BASE, EMAIL, PASSWORD, GLN)
client.authenticate(verbose=True)

# ----------------------------
# 3) Ajohaarojen logiikka
# ----------------------------
if RUN_MODE == "full":
    # --- 3A: KOKO KANTA ---
    print(">>> FULL-ajo: hae kaikki avaimet ja kaikki tuotteet")

    # All → Bronze
    all_keys = fetch_all_keys_to_bronze(spark, client, DELTA_KEYS, batch_size=KEY_BATCH_SIZE)
    print(f"Avaimien (All) nouto onnistui, avaimia talletettu: {all_keys}")

    # Bronze → Silver (kaikki Id:t) — overwrite
    n_fetched = fetch_items_to_silver_json(
        spark,
        client,
        silver_path=DELTA_PRODUCTS,
        bronze_path=DELTA_KEYS,
        batch_size=ITEM_BATCH,
        first_write_mode="overwrite",
        rate_limit_per_minute=RATE_LIMIT,
        verbose=True,
        only_gpc_segment_code=ONLY_GPC_SEGMENT_CODE,   # <-- suodatus käyttöön
    )
    print(f"Koko kannan tuotteet haettu ja tallennettu Silveriin. Rivimäärä: {n_fetched}")

else:
    # --- 3B: VAIN MUUTOKSET ---
    print(f">>> CHANGES-ajo: hae muutokset alkaen {SINCE_ISO.split('T', 1)[0]}")

    chg_ids = []
    offset = None
    while True:
        resp_chg = list_keys_changes(client, batch_size=KEY_BATCH_SIZE, since=SINCE_ISO, offset=offset)
        items = resp_chg.get("Items") or []
        got = [it.get("Id") for it in items if it.get("Id")]
        chg_ids.extend(got)

        print(f"Haettu sivu, uusia ID:itä: {len(got)}, yhteensä: {len(chg_ids)}")

        offset = next_offset(resp_chg)
        if not offset:
            break
        time.sleep(1.0)  # pieni tauko sivutuksen väliin

    print(f"Avaimien (Changes) nouto onnistui, avaimia haettu yhteensä: {len(chg_ids)}")

    if chg_ids:
        # Id-lista → Silver (vain nämä) — append
        n_fetched = fetch_items_to_silver_json(
            spark,
            client,
            silver_path=DELTA_PRODUCTS,
            ids=chg_ids,
            batch_size=ITEM_BATCH,
            first_write_mode="append",
            rate_limit_per_minute=RATE_LIMIT,
            verbose=True,
            only_gpc_segment_code=ONLY_GPC_SEGMENT_CODE,  # <-- suodatus käyttöön
        )
        print(f"Muuttuneiden tuotteiden tiedot noudettu ja tallennettu Silveriin. Rivimäärä: {n_fetched}")
    else:
        print("Ei muutoksia haettavaksi.")

# ----------------------------
# 4) Pikavilkaisu Silveriin (Pandas)
# ----------------------------
silver_df = spark.read.format("delta").load(DELTA_PRODUCTS)
print("Silver-taulun skeema:")
silver_df.printSchema()

print("Esimerkkirivejä Silveristä (Pandas):")
cols = [c for c in ["Id", "ingest_ts", "raw_json"] if c in silver_df.columns]
pdf_silver = silver_df.select(*cols).limit(5).toPandas()

# lyhennetään raw_json esikatseluun
if "raw_json" in pdf_silver.columns:
    pd.set_option("display.max_colwidth", 200)
    pdf_silver["raw_json"] = pdf_silver["raw_json"].str.slice(0, 200)

print(pdf_silver)

# ----------------------------
# 5) Kuratoi → Curated
# ----------------------------
print(">>> Vaihe: Kuratoi tuotteet ja tallenna Deltaan")
curate_mode = "overwrite" if RUN_MODE == "full" else "append"

rows = kuratoi_ja_talleta_deltaan_like_batch(
    spark,
    silver_path=DELTA_PRODUCTS,
    curated_path=CURATED_ITEMS,
    write_mode=curate_mode,
    sample_rows=5
)

print(f"Kuratoituja rivejä kirjoitettu: {rows}")

# ----------------------------
# 6) Pikavilkaisu Curatediin (Pandas)
# ----------------------------
curated_df = spark.read.format("delta").load(CURATED_ITEMS)
cols_curated = ["Id", "BrandName", "TradeItemDescription_fi", "GTIN", "PrimaryImageUrl"]
cols_curated = [c for c in cols_curated if c in curated_df.columns]

print("Esimerkkirivejä Curatedista (Pandas):")
pdf_curated = curated_df.select(*cols_curated).limit(5).toPandas()
print(pdf_curated)


In [0]:
# 1) Lue Curated Delta
curated_df = spark.read.format("delta").load(CURATED_ITEMS)

# (valinnainen) pidä vain uusin rivi per Id LastUpdatedDateTimen perusteella
from pyspark.sql import functions as F, Window
w = Window.partitionBy("Id").orderBy(F.col("LastUpdatedDateTime").desc_nulls_last())
latest_df = (curated_df
             .withColumn("rn", F.row_number().over(w))
             .filter(F.col("rn") == 1)
             .drop("rn"))

to_write = latest_df   # vaihda halutessa -> curated_df, jos haluat viedä kaiken

# 2) JDBC-yhteyden tiedot (kuvan mukaiset)
sql_server = "lejosazure"                      # ilman .database.windows.net -päätettä
sql_db     = "azuredatabase"
jdbc_url   = (
    f"jdbc:sqlserver://{sql_server}.database.windows.net:1433;"
    f"database={sql_db};encrypt=true;trustServerCertificate=false;loginTimeout=30;"
)

username = "azureadmin"
password = "Kesäkuu2023!"

# 3) Kirjoita SQL-tauluun (luo taulun jos sitä ei vielä ole)
#    "append" = lisää perään (muutosajot)
#    "overwrite" = korvaa koko taulun (full-ajo)
(to_write.write
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "dbo.CuratedItems")  # taulun nimi SQL:ssä
    .option("user", username)
    .option("password", password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .mode("append")                         # vaihda "overwrite" jos haluat täyspäivityksen
    .save())

print("✅ Curated-items viety Azure SQL:ään.")

# 4) Pikacheck: hae muutama rivi takaisin SQL:stä
sample = (spark.read.format("jdbc")
          .option("url", jdbc_url)
          .option("query", "SELECT TOP 5 Id, BrandName, TradeItemDescription_fi, GTIN, PrimaryImageUrl FROM dbo.CuratedItems ORDER BY Id")
          .option("user", username)
          .option("password", password)
          .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
          .load()).toPandas()

print(sample)

